# ST 554 Project 2 

### By Hannah Shaw

## Introduction

In this project, we will be using spark SQL and python to read in a CSV file and convert it into a data frame in order to check and clean the data. To do so, we will design and implement a data quality class in pyspark. Instead of writing a Spark script with standard spark functionality, however, we will instead create a python class that wraps a Spark (SQL style) DataFrame and provides functionality for cleaning and checking the data. 

In the first part of this project, we will create a class in a separate .py file called `SparkDataCheck` that will work on Spark SQL style data frames.

In the second part of the project, we will analyze another dataset using both spark SQL style data frames and the pandas-on-spark data frames to practice more with our created class. 

## Part I: Creating a Class

After creating our `SparkDataCheck` class in `SparkDataCheck.py`, we will import our class (along with the necessary modules and functions) into this notebook.

In [2]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import numpy as np
import pandas as pd

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("myapp").config("spark.sql.ansi.enabled", "false").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 23:40:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 23:40:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/26 23:40:35 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/26 23:40:35 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/03/26 23:40:35 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [ ]:
#import our class
import SparkDataCheck

In [ ]:
import importlib #this lets us reload the individual module above after we've editted SparkDataCheck
importlib.reload(SparkDataCheck)

Now that we've imported our `SparkDataCheck` class, we want to test it by reading in the air quality data we used in the first project, which is available at https://www4.stat.ncsu.edu/online/datasets/air.csv. We will read in the air quality csv, and then use one of `SparkDataCheck`'s methods to create an instance of the class from this csv file.

## Part II

We will now use our SparkDataCheck class to perform basic analysis using spark on some NFL data. We will first do this using solely pandas-on-Spark, and then repeat the same process using only the Spark SQL DataFrame.

First, we read in the CSV file for the weekly NFL data and view the first 5 rows of the data frame.

In [5]:
nfl = pd.read_csv("weekly_nfl_data.csv")
nfl.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,...,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,MIA,1999,1,REG,...,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,MIA,1999,2,REG,...,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,MIA,1999,4,REG,...,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,CLE,1999,7,REG,...,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,NaN,Abdul-Karim al-Jabbar,RB,RB,NaN,CLE,1999,8,REG,...,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


We can see all of the column names here:

In [7]:
nfl.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

We want to only look at quarterback stats for the seasons 2005 to 2023 (inclusive). Thus, we will subset the rows of the data to only include the position "QB", the regular season (“REG”), and the years in the range noted. We will also subset the columns to only include `player_display_name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`.

For each `player_display_name` and `season` combination, we will now find the sum and mean of each of the statistical quantities (the rest of the columns we chose above).

We will also create two new various (by season/player combination):
 - `completion_percentage` = (sum of completions)/(sum of attempts), i.e. what percentage of attempted throws resulted in completions.
 - `td_int_ratio` = (sum passing tds)/(sum interceptions), the ratio of touchdowns versus interceptions.